In [ ]:
import gymnasium as gym
from my_USV_Env import MyUsvEnv_Gym
# Register the environment so we can create it with gym.make()
gym.register(
    id="myenv/MyUsvEnv_Gym-v0",
    entry_point=MyUsvEnv_Gym,
)
# 2. 创建环境实例
env = gym.make("myenv/MyUsvEnv_Gym-v0")  # 使用注册的id创建环境
from gymnasium.utils.env_checker import check_env
# This will catch many common issues
try:
    check_env(env)
    print("Environment passes all checks!")
except Exception as e:
    print(f"Environment has issues: {e}")

In [ ]:
from stable_baselines3 import TD3
from stable_baselines3.common.env_util import make_vec_env
# Parallel environments
vec_env = make_vec_env("myenv/MyUsvEnv_Gym-v0", n_envs=4)
model = TD3("MultiInputPolicy", vec_env, verbose=1)
model.learn(total_timesteps=250000)
model.save("td3_MyUsvEnv")

In [ ]:
# 创建测试环境（使用与训练相同的环境参数）
from stable_baselines3 import TD3
import gymnasium as gym
import matplotlib.pyplot as plt
from my_USV_Env import MAX, MyUsvEnv_Gym
model = TD3.load("td3_MyUsvEnv")

# Register the environment so we can create it with gym.make()
gym.register(
    id="myenv/MyUsvEnv_Gym-v0",
    entry_point=MyUsvEnv_Gym,
)
env = gym.make("myenv/MyUsvEnv_Gym-v0")
env.unwrapped.__init__(record_flag=True) 
obs, info = env.reset()  # 新版Gym返回元组
terminated = False
truncated = False
total_reward = 0

while not (terminated or truncated):
    action, _ = model.predict(obs, deterministic=True)  # 确保传入的是numpy数组
    # 新版Gym返回5个值
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    points=(float(obs["x_norm"]*MAX),float(obs["y_norm"]*MAX))
    # env.record_points(points)
    env.unwrapped.record_points(points,csv_file="usv_path_points_2.csv")
print(total_reward)    
env.unwrapped.draw_path("usv_path_points_2.csv")




In [ ]:
from netCDF4 import Dataset

# 替换为你的文件路径
file_path = "data\currents_reduced.nc"

# 打开 NetCDF 文件
nc_data = Dataset(file_path, "r")

# 打印所有变量列表
print("Variables in the file:")
print(nc_data.variables.keys())

# 打印文件全局属性（如单位、坐标系等）
print("\nGlobal attributes:")
print(nc_data.__dict__)



In [ ]:
from netCDF4 import Dataset
import numpy as np

def extract_currents(input_path, output_path, time_step=0, depth_level=0):
    """分块提取指定时间层和深度层的流速数据"""
    with Dataset(input_path, "r") as src:
        # 获取维度信息
        time_dim = len(src['time'])
        depth_dim = len(src['depth'])
        lat_dim = len(src['latitude'])
        lon_dim = len(src['longitude'])
        
        with Dataset(output_path, "w", format="NETCDF4") as dst:
            # 设置压缩参数
            comp = dict(zlib=True, complevel=4)
            
            # 创建维度
            for dim in ['time', 'depth', 'latitude', 'longitude']:
                dst.createDimension(dim, len(src.dimensions[dim]))
            
            # 创建坐标变量
            for var in ['time', 'depth', 'latitude', 'longitude']:
                src_var = src[var]
                dst_var = dst.createVariable(var, src_var.datatype, (var,), **comp)
                dst_var[:] = src_var[:]
                dst_var.setncatts({k: src_var.getncattr(k) for k in src_var.ncattrs()})
            
            # 创建流速变量（仅保留指定层）
            for var in ['uo', 'vo']:
                src_var = src[var]
                chunks = (1, 1, lat_dim, lon_dim)  # 分块参数
                
                # 创建4D变量但只写入指定层
                dst_var = dst.createVariable(var, np.float32, 
                                           ('time', 'depth', 'latitude', 'longitude'),
                                           chunksizes=chunks, **comp)
                
                # 逐时间步写入
                for t in range(time_dim):
                    # 分块读取数据（按时间步）
                    data = src_var[t, depth_level, :, :].astype(np.float32)
                    dst_var[t, depth_level, :, :] = data

if __name__ == "__main__":
    input_path = "data/globalsea_20250901.nc"
    output_path = "data/currents_reduced.nc"
    
    extract_currents(input_path, output_path,
                   time_step=1,  # 只保留第一个时间步
                   depth_level=1)  # 只保留表层（depth=0）


In [ ]:
from netCDF4 import Dataset
import numpy as np
file_path = "data\globalsea_20250901.nc"

try:
    with Dataset(file_path, "r") as nc_data:
        # 检查变量名
        print("Variables:", nc_data.variables.keys())
        
        # 提取经纬度（根据实际变量名修改）
        lon = nc_data.variables["longitude"][3600:3700]
        lat = nc_data.variables["latitude"][1248:1348]
        uu = nc_data.variables['uo'][0, 0,:,:][1248:1348, 3600:3700]   # 经度方向速度
        vv = nc_data.variables['vo'][0, 0,:,:][1248:1348, 3600:3700]   # 纬度方向速度
        
        # 计算范围
        print(f"经度范围: {lon.min()} ~ {lon.max()}")
        print(f"纬度范围: {lat.min()} ~ {lat.max()}")

        print(lon.shape, lat.shape)
        print(uu.shape, vv.shape)

        print(f"速度范围: {uu.min()} ~ {uu.max()}, {vv.min()} ~ {vv.max()}")
except Exception as e:
    print(f"读取文件失败: {str(e)}")



In [ ]:
# from real2D_Env import Env
from my_USV_Env import MyUsvEnv
env= MyUsvEnv()
env.reset()
env.visualize_env()